In [2]:
import sys
import os

print("☢️ INITIATING NUCLEAR CLEANUP...")

# Uninstall EVERYTHING related to the problem
libraries = [
    "tensorflow", "tensorflow-intel", "tensorflow-estimator", "keras", 
    "tensorboard", "numpy", "pandas", "scipy", "mne", "h5py", "matplotlib"
]

for lib in libraries:
    os.system(f"{sys.executable} -m pip uninstall -y {lib}")

print("✅ Cleanup Complete. The slate is clean.")

☢️ INITIATING NUCLEAR CLEANUP...
Found existing installation: tensorflow 2.15.0
Uninstalling tensorflow-2.15.0:
  Successfully uninstalled tensorflow-2.15.0


Found existing installation: tensorflow-estimator 2.15.0
Uninstalling tensorflow-estimator-2.15.0:
  Successfully uninstalled tensorflow-estimator-2.15.0


Found existing installation: keras 2.15.0
Uninstalling keras-2.15.0:
  Successfully uninstalled keras-2.15.0


Found existing installation: tensorboard 2.15.2
Uninstalling tensorboard-2.15.2:
  Successfully uninstalled tensorboard-2.15.2


Found existing installation: numpy 1.25.2
Uninstalling numpy-1.25.2:
  Successfully uninstalled numpy-1.25.2


Found existing installation: pandas 2.0.3
Uninstalling pandas-2.0.3:
  Successfully uninstalled pandas-2.0.3


Found existing installation: scipy 1.11.4
Uninstalling scipy-1.11.4:
  Successfully uninstalled scipy-1.11.4


Found existing installation: mne 1.6.0
Uninstalling mne-1.6.0:
  Successfully uninstalled mne-1.6.0


Found existing installation: h5py 3.10.0
Uninstalling h5py-3.10.0:
  Successfully uninstalled h5py-3.10.0


Found existing installation: matplotlib 3.10.8
Uninstalling matplotlib-3.10.8:
  Successfully uninstalled matplotlib-3.10.8
✅ Cleanup Complete. The slate is clean.


In [3]:
import sys

print("🛠️ Installing the 'Time Capsule' (Stable 2023 Stack)...")

# We install specific versions that are guaranteed to like each other
!{sys.executable} -m pip install tensorflow==2.12.0 numpy==1.23.5 pandas==1.5.3 scipy==1.10.1 mne==1.3.1 matplotlib h5py==3.8.0

print("✅ Installation Complete.")

🛠️ Installing the 'Time Capsule' (Stable 2023 Stack)...
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
  Obtaining dependency information for tensorflow==2.12.0 from https://files.pythonhosted.org/packages/3f/b2/33372601ed71fb41049642f8f6e1e142215e8b5c3463df434fc8885db278/tensorflow-2.12.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
  Obtaining dependency information for numpy==1.23.5 from https://files.pythonhosted.org/packages/e4/f3/679b3a042a127de0d7c84874913c3e23bb84646eb3bc6ecab3f8c872edc9/numpy-1.23.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
  Obtaining dependency information for pandas==1.5.3 from https://files.pythonhosted.org/packages/49/e2/79e46612dc25ebc7603dc11c560baa7266c90f9e48537ecf1a02a0dd6bff/pandas-1.5.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
  Obtaining dependency information for scipy==1.10.1 from https://files.pythonhosted.org/packages/1f/4b/3bacad9a166350cb2e5

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import mne
from scipy import signal

# ==========================================
# CONFIGURATION
# ==========================================
EDF_FOLDER = "raw_data"             # Where your downloaded files are
OUTPUT_FOLDER = "dataset_spectrograms"    # Where images will be saved
WINDOW_SECONDS = 4
FS = 256                            # Sampling Frequency of CHB-MIT (256 Hz)

# Create the output folders automatically
os.makedirs(f"{OUTPUT_FOLDER}/healthy", exist_ok=True)
os.makedirs(f"{OUTPUT_FOLDER}/seizure", exist_ok=True)

print("Block 1 Complete: Libraries loaded and folders created.")

Block 1 Complete: Libraries loaded and folders created.


In [2]:
# 1. What is FS = 256? (The "Dots per Second")When the EEG machine records your brain, it doesn't draw a smooth line. It actually saves a long list of numbers (dots).FS = 256 simply means the machine saves 256 dots for every 1 second of time.If you record for 10 seconds, you have 2,560 dots ($256 \times 10$).Why the code needs it:Imagine I give you a list of 1,000 dots and ask, "Is this a fast wave or a slow wave?"You cannot answer unless you know how much time those dots represent.If those 1,000 dots happened in 1 second, it's a super fast wave.If those 1,000 dots happened in 1 hour, it's a super slow wave.By telling the code FS = 256, you are giving it the ruler to measure time. It now knows: "Okay, every 256 dots is exactly one second." Now it can calculate the frequency correctly.

In [3]:
# ==========================================
# THE SEIZURE "ANSWER KEY"
# Format: 'filename.edf': [(start_time, end_time), (start2, end2)]
# ==========================================
SEIZURE_TIMES = {
    # Patient 01
    "chb01_03.edf": [(2996, 3036)],
    "chb01_04.edf": [(1467, 1494)],
    "chb01_15.edf": [(1732, 1772)],
    "chb01_16.edf": [(1015, 1066)],
    "chb01_18.edf": [(1720, 1810)],
    "chb01_26.edf": [(1862, 1963)],

    # Patient 02
    "chb02_16.edf": [(130, 212), (2972, 3053)],
    "chb02_19.edf": [(3369, 3378)],

    # Patient 03
    "chb03_01.edf": [(362, 414)],
    "chb03_02.edf": [(731, 796)],

    # Patient 05 & 08
    "chb05_13.edf": [(3462, 3476)],
    "chb08_11.edf": [(2677, 2682)]
}

print("Block 2 Complete: Seizure times loaded.")

Block 2 Complete: Seizure times loaded.


In [4]:
def save_spectrogram(data, label, filename, index):
    #  # SAVE LOGIC:
#         if is_seizure == 1 or (i % 10 == 0):
#             segment = data[start:end]
#             save_spectrogram(segment, is_seizure, file_name.replace('.edf',''), i)
    
    #Generates a Spectrogram (Heatmap) and saves it as a PNG.
    # 1. Compute the Spectrogram (Math -> Frequency)
    # nperseg=128 gives us a good balance of time vs frequency resolution
    f, t, Sxx = signal.spectrogram(data, fs=FS, nperseg=128, noverlap=64)
# signal.spectrogram(...):  #This is a function from the Scipy library that performs a Fourier Transform.
#It takes a squiggly line (time vs. voltage) grasph of our .edf file and breaks it down into frequencies. -->FROM EDF GRAPH
#data: The raw brainwave numbers (e.g., array of 1024 numbers for 4 seconds).
#fs=FS: The Sampling Frequency (256 Hz). This tells the math function that "256 numbers = 1 second."

#Output (Sxx): This is a 2D grid of numbers representing "Loudness" at different frequencies
# #f and t are simply the labels for the axes of your graph.                                         -->TO SPECTROGRAM GRAPH
# 1. f = Frequency (The Y-Axis)
# What it is: An array of numbers representing the "pitch" of the brainwave (e.g., 0Hz, 10Hz, 20Hz... up to 128Hz).
# Role: It tells the plotter how "high" to draw the pixels.
# 2. t = Time (The X-Axis)
# What it is: An array of time steps (e.g., 0.0s, 0.1s, 0.2s... up to 4.0s).


    # 2. Plotting the Image
    # We create a small 128x128 pixel image (dpi=100, figsize=1.28)
    fig = plt.figure(figsize=(1.28, 1.28), dpi=100)
    #plt.figure: Opens a blank canvas in memory.
    #figsize=(1.28, 1.28): We set the size to be tiny—1.28 inches by 1.28 inches.
# dpi=100: "Dots Per Inch."Math: $1.28 \text{ inches} \times 100 \text{ dpi} = 128 \text{ pixels}$.
# This ensures our final image is exactly 128x128 pixels, which is the perfect size for our CNN model later.


    # 'cmap=jet' makes the seizure (high energy) look Red/Yellow
    # We add 1e-10 to avoid log(0) errors
    plt.pcolormesh(t, f, 10 * np.log10(Sxx + 1e-10), shading='gouraud', cmap='jet')
#plt.pcolormesh: This function colors the grid.
#cmap='jet': The "Color Map." This tells Python:
# Blue = Quiet/Low Energy.
# Red/Yellow = Loud/High Energy (Seizure!).
#shading smoothens the grid
    
    
    
    plt.axis('off') # Remove ruler/numbers (AI doesn't need them)

    # 3. Save to the correct folder
    class_folder = "seizure" if label == 1 else "healthy"
    save_path = f"{OUTPUT_FOLDER}/{class_folder}/{filename}_{index}.png"

    plt.savefig(save_path, bbox_inches='tight', pad_inches=0)
    plt.close(fig) # Close the figure immediately to free up RAM

print("Block 3 Complete: Function defined.")

Block 3 Complete: Function defined.


In [5]:
'''
nperseg=128: "Take a photo that is 128 pixels wide." (This determines vertical resolution).
1. The "Time" Reason (0.5 Seconds)Your Sampling Rate (FS) is 256 Hz.This means the machine records 256 numbers every 1 second.
We chose 128.$128 / 256 = 0.5 \text{ seconds}$.Why is this good?Seizure spikes are very fast. 
If we took a huge window (like 1000 numbers), the tiny spike would get lost in the crowd.
By looking at 0.5 seconds at a time, we are looking at a "snapshot" that is short enough to catch a quick spike, 
but long enough to see a pattern
'''








'''
Let's try a visual approach. Imagine you are looking at a long train moving past a window.

### **The Concept: The Sliding Magnifying Glass**

You have a long strip of brain signal. You cannot look at the whole thing at once. You have a "Magnifying Glass" (the window) that can only see **128 points** at a time.

#### **Scenario A: No Overlap (Bad)**

If you move the glass 128 steps every time, you touch the edges perfectly.

* **Look 1:** You see points 0 to 128.
* **Look 2:** You jump to points 128 to 256.

**The Problem:** What if the Seizure happens exactly at point **128**?

* "Look 1" sees the *start* of the seizure but cuts it off.
* "Look 2" sees the *end* of the seizure but misses the start.
* **Result:** The AI gets two "half-pictures" and might fail to recognize either one. It's like cutting a photograph of a face in half—it's hard to recognize the person.

---

#### **Scenario B: With Overlap (Good)**

This is what `noverlap=64` does. It means "Copy the last 64 points into the next picture."

Let's trace the numbers:

1. **First Look:** Points **0 to 128**.
2. **Slide:** Instead of jumping to 128, we only slide forward by **64 steps**.
3. **Second Look:** Points **64 to 192**.

**Why this fixes the problem:**
Remember the seizure at point **128**?

* In the **First Look**, it was on the *edge*.
* But in the **Second Look** (64 to 192), point 128 is right in the **middle** (center).
* **Result:** The AI gets a perfect, centered picture of the seizure in the second frame.

### **Simple ASCII Diagram**

Imagine your signal is this line of dashes:
`--------------------------------`

**Without Overlap:** (Gaps at the edges)
`[      Window 1      ][      Window 2      ]`

**With Overlap:** (Everything is covered twice)
`[      Window 1      ]`
`        [      Window 2      ]`
`                [      Window 3      ]`

See how Window 2 "covers the gap" between Window 1 and 3? That is exactly what `noverlap` does. It ensures nothing gets lost in the cracks.
'''

'\nLet\'s try a visual approach. Imagine you are looking at a long train moving past a window.\n\n### **The Concept: The Sliding Magnifying Glass**\n\nYou have a long strip of brain signal. You cannot look at the whole thing at once. You have a "Magnifying Glass" (the window) that can only see **128 points** at a time.\n\n#### **Scenario A: No Overlap (Bad)**\n\nIf you move the glass 128 steps every time, you touch the edges perfectly.\n\n* **Look 1:** You see points 0 to 128.\n* **Look 2:** You jump to points 128 to 256.\n\n**The Problem:** What if the Seizure happens exactly at point **128**?\n\n* "Look 1" sees the *start* of the seizure but cuts it off.\n* "Look 2" sees the *end* of the seizure but misses the start.\n* **Result:** The AI gets two "half-pictures" and might fail to recognize either one. It\'s like cutting a photograph of a face in half—it\'s hard to recognize the person.\n\n---\n\n#### **Scenario B: With Overlap (Good)**\n\nThis is what `noverlap=64` does. It means "C

In [6]:
# Think of this like listening to music.

# ### **1. The Problem: "Whisper vs. Jet Engine"**

# Brain signals have a massive range of loudness.

# * **Normal thinking** is like a whisper (Energy = `0.001`).
# * **A Seizure** is like a jet engine (Energy = `1000`).

# If you try to draw both on the same graph without math, you have a problem:

# * If you zoom out to see the Jet Engine (1000), the Whisper (0.001) is so small it becomes **invisible**. It just looks like zero.
# * You would miss all the important "quiet" details of the brain because the seizure squashed them.

# ### **2. The Solution: The Logarithm (`np.log10`)**

# The Logarithm is a "Fairness Filter." It squashes huge numbers down and lifts tiny numbers up so you can see **both** at the same time.

# Look at how the math changes the numbers:

# | Signal Type | Raw Energy (Linear) | Math: `10 * log10(Energy)` | Result |
# | --- | --- | --- | --- |
# | **Seizure** | `1000` | `10 * 3` | **30** |
# | **Normal** | `0.001` | `10 * -3` | **-30** |

# **The Magic:**

# * Before: The difference was massive (1000 vs 0.001).
# * After: The difference is manageable (30 vs -30).
# * Now, on your colorful heatmap, the Seizure is **Red (30)** and the Normal brain is **Blue (-30)**. Both are clearly visible.

# ---

# ### **3. The "Safety Crumb" (`+ 1e-10`)**

# This is purely to stop your code from crashing.

# * **The Math Rule:** You literally cannot calculate the Log of Zero.
# *  (Negative Infinity).


# * **The Scenario:** Sometimes, for a split second, the brain signal might be perfectly flat (Energy = `0`).
# * **The Crash:** If Python tries to do `np.log10(0)`, it panics and gives you an error or a broken value called `NaN` (Not a Number).

# **The Fix:**
# We lie to the computer just a tiny bit. Instead of zero, we give it `0.0000000001` (that is `1e-10`).

# * Math: `np.log10(0 + 0.0000000001)` = `-100`.
# * Result: The computer is happy, creates a very dark blue pixel (meaning "very quiet"), and the code keeps running without crashing.

In [7]:
# if label == 1: A simple switch. If the window was marked as a seizure, put it in the "seizure" folder. Otherwise, put it in "healthy".

# bbox_inches='tight', pad_inches=0: Even after axis('off'), Python sometimes leaves a white empty border around the image. 
# This command forcefully crops it tight so every single pixel is data.

# plt.close(fig): Crucial. If you don't do this, Python keeps the image open in RAM. After 10,000 images, your computer would crash. 
# This deletes the image from RAM after saving it to the disk.

In [8]:
# ==========================================
# MAIN PROCESSING LOOP (WITH NOTCH FILTER)
# ==========================================
print("--- STARTING IMAGE GENERATION ---")

# Get list of all .edf files in the raw_data folder
files = [f for f in os.listdir(EDF_FOLDER) if f.endswith('.edf')]

for file_name in files:
    print(f"Processing {file_name}...")
    file_path = os.path.join(EDF_FOLDER, file_name)

    try:
        # 1. Load the file
        raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)

        # 2. NEW: Notch Filter (Removes 60Hz US Power Line Noise)
        # This kills the "buzz" from the hospital equipment
        raw.notch_filter(freqs=60, method='fir', verbose=False)

        # 3. Bandpass Filter (Keep frequencies between 1Hz and 70Hz)
        raw.filter(1., 70., verbose=False)

    except Exception as e:
        print(f"Could not read {file_name}: {e}")
        continue

    # We select Channel 0 (usually FP1-F7) for simplicity
    data = raw.get_data()[0] * 1e6

    # Calculate how many 4-second windows fit in this file
    n_windows = len(data) // (WINDOW_SECONDS * FS)

    # Get the seizure times for THIS specific file
    seizures = SEIZURE_TIMES.get(file_name, [])

    for i in range(n_windows):
        start = i * WINDOW_SECONDS * FS
        end = start + (WINDOW_SECONDS * FS)

        # Check if this 4-second clip falls inside a seizure time
        window_mid_sec = (start + end) / 2 / FS
        is_seizure = 0

        for (s_start, s_end) in seizures:
            if s_start <= window_mid_sec <= s_end:
                is_seizure = 1
                break

        # SAVE LOGIC:
        if is_seizure == 1 or (i % 10 == 0):
            segment = data[start:end]
            save_spectrogram(segment, is_seizure, file_name.replace('.edf',''), i)

print("Block 4 Complete: All images generated (Cleaned with Notch Filter)!")

--- STARTING IMAGE GENERATION ---
Processing chb03_01.edf...


/tmp/ipykernel_27153/3912208888.py:15: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)


Processing chb03_02.edf...


/tmp/ipykernel_27153/3912208888.py:15: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)


Processing chb05_13.edf...


...

Processing chb08_11.edf...


/tmp/ipykernel_27153/3912208888.py:15: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)


Processing chb01_16.edf...


/tmp/ipykernel_27153/3912208888.py:15: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)


Block 4 Complete: All images generated (Cleaned with Notch Filter)!


In [ ]:
# raw.get_data(): This grabs the huge grid of numbers from the file.[0]: The file has 23 channels (rows). 
#         We only want the first one (Channel 0) to keep things simple for the AI right now.* 1e6: 
#             This multiplies every number by 1,000,000.Why? Brain signals are tiny (e.g., 0.00005 Volts). 
#             Computers hate tiny decimals. 
#             This converts it to Microvolts (e.g., 50 $\mu V$), which are much easier to handle.

In [ ]:
# This calculates: "How many slices of cake can I get from this whole cake?"len(data):
# Total number of dots (samples) in the file.
# WINDOW_SECONDS * FS: How many dots make up 4 seconds ($4 \times 256 = 1024$ dots).

In [ ]:
# # 3. The "Mid-Window" Solution (The Safety Check)
# # window_mid_sec = (start + end) / 2 / FS
# But consider the opposite: Imagine the seizure ends just as the window starts.

# Seizure ends at 0.1s.

# Window is 0.0s to 4.0s.

# Check Start (0.0s): It is inside the seizure!

# Result: You label this mostly healthy image as "SEIZURE."

# 2. Why the CNN Hates Ambiguity
# The Random Forest uses math (numbers). It can handle a little bit of noise.

# But a CNN is looking at a picture. If you show the CNN a picture that looks 95% healthy, but you label it "Seizure" 
#(just because a seizure started at the very last pixel), the CNN gets extremely confused. 
#It starts thinking, "Wait, healthy waves are seizures now?"
# # By checking the middle of the window (the 2-second mark), you are forcing the computer to be strict.

# # It says: "I will only call this a Seizure if the seizure is happening right in the center of the image."

# # This ensures that the seizure is the main character of that picture, not just a tiny photobomb at the edge.

# # In Summary:

# # Start Time Check (RF): Good for simple math features. Fast but slightly inaccurate at the edges.

# # Mid Time Check (CNN): Ensures the image contains a strong, clear seizure pattern so the AI learns the right visual shapes. It improves the quality of your training images.

In [ ]:
# You have spotted a **very real flaw** in the logic. This is an advanced observation—most students miss this!

# You are asking: **"What if a short seizure happens exactly on the border between two windows?"**

# Let's look at the "Blind Spot" you found:

# * **Window 1:** 0s to 4s. (Midpoint: **2s**)
# * **Window 2:** 4s to 8s. (Midpoint: **6s**)
# * **The Scenario:** Imagine a short seizure starts at **3s** and ends at **5s**.

# **The Failure:**

# 1. **Window 1 Check:** Is the midpoint (2s) inside the seizure (3s-5s)? **NO.** (Labeled Healthy)
# 2. **Window 2 Check:** Is the midpoint (6s) inside the seizure (3s-5s)? **NO.** (Labeled Healthy)

# **Result:** You completely missed the seizure. It fell into the "crack" between the midpoints.

# ---

# ### Why did we write the code this way?

# There are two reasons, one for **Training** (Step 2) and one for **Testing** (Step 3).

# #### 1. For Training: We want "Textbook Examples"

# When you are teaching a child (or an AI) what a cat looks like, you don't show them a photo of *half a cat* disappearing off the edge of the picture. You show them a cat right in the center.

# * By checking the **Midpoint**, we are intentionally being picky.
# * We are saying: *"If the seizure is just a tiny slice on the edge, IGNORE IT. I only want to train on images where the seizure is the main event."*
# * **Benefit:** This makes your training data "cleaner" and easier for the AI to learn.
# * **Cost:** We lose a few "edge case" examples (like the 3s-5s one). Since CHB-MIT seizures are usually long (40 seconds to minutes), losing 2 seconds of data is acceptable to get higher quality images.

# #### 2. The Solution for Real Life: "Overlapping Windows"

# In a real hospital setting (and often in Step 3 Testing), we solve your fear by changing the **Stride** (the step size).

# Instead of jumping 4 seconds (0-4, 4-8, 8-12), we slide the window by **1 second** (Overlapping).

# * **Window A:** 0s - 4s (Midpoint: 2s) -> Misses seizure.
# * **Window B:** 1s - 5s (Midpoint: 3s) -> **CATCHES IT!** (3s is inside 3s-5s).
# * **Window C:** 2s - 6s (Midpoint: 4s) -> **CATCHES IT!**

# **Summary:**

# * **In Step 2 (Training):** We use **Non-Overlapping** (Jump 4s) to keep the dataset small and clean. We accept the risk of missing tiny border seizures because we have plenty of long seizures to learn from.
# * **In Step 3 (Real Deployment):** If you are worried about missing seizures, you simply change the loop to step forward by `1 second` instead of `4 seconds`. This closes the gap completely.

In [ ]:
# # EXPLAIN TRHIS PS
# #  # SAVE LOGIC:
# #         if is_seizure == 1 or (i % 10 == 0):
# #             segment = data[start:end]
# #             save_spectrogram(segment, is_seizure, file_name.replace('.edf',''), i)


# This snippet is a Data Balancer. It is one of the most important lines in your entire project because it prevents your AI from becoming "lazy."

# Here is the plain English translation of that if statement:

# "If this image contains a Seizure, SAVE IT. If it is Healthy, only save it 10% of the time (throw the rest away)."


# Rule 1: The VIP Rule (if is_seizure == 1)
# Logic: If the data shows a seizure, we keep it every single time.

# Why: Seizures are rare "gold dust." We cannot afford to lose a single second of seizure data.
    
    
# Rule 2: The "1-in-10" Rule (or i % 10 == 0)
# Logic: This part handles the Healthy (Normal) data.

# The Math: i % 10 == 0 uses the Modulo operator. It checks if the window number (i) divides perfectly by 10 (e.g., window 0, 10, 20, 30...).

# The Result: It skips windows 1 through 9, saves window 10, skips 11 through 19, saves 20, and so on.

# Why: You have too much healthy data. If a patient is recorded for 24 hours, they might be healthy for 23 hours and 55 minutes. If you saved all of that, your dataset would be 99% healthy images.